# Test Bronze — WHO
Verifica las 2 tablas Bronze de WHO (Volume + Google Drive).  
Usa `limit(10)` — no ejecuta full scan.

In [0]:
from pyspark.sql import functions as F

BRONZE_SCHEMA = "sandbox_bronze_ss2"
_TABLES = ["who_guatemala", "who_costa_rica"]

# Columnas de datos esperadas (nombres reales en Bronze — en inglés, del CSV de la OMS)
_KEY_DATA_COLS = ["indicator_code", "indicator_name", "year", "sex", "age_group_code", "number"]
# Auditoría común a ambas tablas
_AUDIT_COMMON  = ["ingestion_timestamp", "source_system", "source_file", "country"]
# Auditoría específica por fuente
_AUDIT_GT      = ["volume_path"]    # who_guatemala (Databricks Volume)
_AUDIT_CR      = ["gdrive_file_id"] # who_costa_rica (Google Drive)

---
## who_guatemala

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.who_guatemala")
df.printSchema()

In [0]:
display(df.limit(10))

---
## who_costa_rica

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.who_costa_rica")
df.printSchema()

In [0]:
display(df.limit(10))

---
## Chequeos de calidad (sin full scan)

In [0]:
# 1. Columnas de datos y auditoría presentes
print("── Columnas presentes ──")
_AUDIT_POR_TABLA = {
    "who_guatemala": _AUDIT_COMMON + _AUDIT_GT,
    "who_costa_rica": _AUDIT_COMMON + _AUDIT_CR,
}
for t in _TABLES:
    cols = spark.table(f"{BRONZE_SCHEMA}.{t}").columns
    esperadas = _KEY_DATA_COLS + _AUDIT_POR_TABLA[t]
    missing = [c for c in esperadas if c not in cols]
    print(f"  {t}: {'OK' if not missing else f'WARN — faltan: {missing}'}")

In [0]:
# 2. Nulls en columnas clave (muestra 1k)
print("── Nulls en columnas clave (muestra 1k) ──")
for t in _TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_presentes = [c for c in _KEY_DATA_COLS if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_presentes:
        nulls = sample.filter(F.col(col).isNull()).count()
        print(f"    {col} nulls: {nulls}")

In [0]:
# 3. Auditoría sin nulls
print("── Nulls en auditoría (muestra 1k) ──")
for t in _TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(1000)
    cols_audit = [c for c in _AUDIT_POR_TABLA[t] if c in sample.columns]
    print(f"\n  {t}")
    for col in cols_audit:
        nulls = sample.filter(F.col(col).isNull()).count()
        status = "OK" if nulls == 0 else f"WARN — {nulls} nulls"
        print(f"    {col}: {status}")

In [0]:
# 4. Valores distintos de country y rango de year
print("── country y rango year (muestra 5k) ──")
for t in _TABLES:
    sample = spark.table(f"{BRONZE_SCHEMA}.{t}").limit(5000)
    paises  = [r.country for r in sample.select("country").distinct().collect()]
    rango   = sample.agg(F.min("year").alias("min"), F.max("year").alias("max")).collect()[0]
    print(f"  {t}: country={paises}  year={rango['min']}—{rango['max']}")